# Set Up

In [ ]:
# mounting drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# imports
import numpy as np
from scipy.optimize import linprog
from time import time
import pandas as pd
import scipy.sparse as sparse

In [ ]:
# Statistics

Nlocations = 16774
current_AED_num = 7926

print("Number of locations:", Nlocations)
print("Twice the number of locations:", 2 * Nlocations)

# Diagonal Demand

In [ ]:
# Importing objective function

objective_fct = pd.read_csv('/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/objective_fct.csv')
objective_fct = objective_fct[["x"]].values
objective_fct = objective_fct.squeeze()

print(objective_fct.shape)
print("Correct shape?",objective_fct.shape == (Nlocations * 2,))


#print(objective_fct)

(33548,)
Correct shape? True


In [ ]:
# Importing RHS

additional = False

if(additional):
    name_ext = "_additional"
    constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_rhs_diag_additional.csv")
else:
    name_ext = ""
    constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_rhs_diag.csv")

constraint_rhs = constraint_rhs[["x"]].values
constraint_rhs = constraint_rhs.squeeze()

print(constraint_rhs.shape)
print("Correct shape?",constraint_rhs.shape == (Nlocations + 1,))

#print(constraint_rhs)

(16775,)
Correct shape? True


In [ ]:
# Importing Constraint Matrix

A_ub = sparse.load_npz("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_mat_diag_sparse.npz")

print(A_ub.shape)
print("Correct shape?",A_ub.shape == (Nlocations , Nlocations * 2))

(16774, 33548)
Correct shape? True


In [ ]:
# resetting limit
pc_to_place = 150

total_limit = np.round(current_AED_num * pc_to_place / 100)
#total_limit = 100 # alternative manual setting

print("AEDs to place:", total_limit)

constraint_rhs[0] = total_limit

AEDs to place: 11889.0


In [ ]:
# Performing linear programming

solution = linprog(c = objective_fct,
                   A_ub = A_ub,
                   A_eq = np.matrix(1-objective_fct),
                   b_ub = - constraint_rhs[1:],
                   b_eq = constraint_rhs[0:1],
                   bounds = (0,15),
                   integrality= 1-objective_fct
                   )

print(solution.message)
print("success:", solution.success)
print("status:", solution.status)

Optimization terminated successfully. (HiGHS Status 7: Optimal)
success: True
status: 0


In [ ]:
# Printing solution - When tiny

print("solution:", solution.x[:Nlocations])

solution: [1. 1. 1. ... 0. 0. 1.]


In [ ]:
# Checking solution

print(solution.message)
print(max(solution.x[:Nlocations])) # should be 3 or lower
print(max(solution.x[Nlocations+1:])) # should be 4 or lower
print("Checking if placed correct number: ", (sum(solution.x[:Nlocations]) - total_limit < 0.01)) # should be true

solution_name = "diagional_sol_" + str(round(total_limit)) + "AEDs_" + str(Nlocations) + "locations" + name_ext

np.savetxt("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_exports/" + solution_name + ".csv", solution.x)

Optimization terminated successfully. (HiGHS Status 7: Optimal)
3.0
0.19375170728774016
Checking if placed correct number:  True


# Gradated Demand

In [ ]:
# Importing objective function

objective_fct = pd.read_csv('/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/objective_fct.csv')
objective_fct = objective_fct[["x"]].values
objective_fct = objective_fct.squeeze()

print(objective_fct.shape)
print("Correct shape?",objective_fct.shape == (Nlocations * 2,))


#print(objective_fct)

(33548,)
Correct shape? True


In [ ]:
# Importing RHS

additional = False

if(additional):
    name_ext = "_additional"
    constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_rhs_grad_additional.csv")
else:
    name_ext = ""
    constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_rhs_grad.csv")

constraint_rhs = constraint_rhs[["x"]].values
constraint_rhs = constraint_rhs.squeeze()

print(constraint_rhs.shape)
print("Correct shape?",constraint_rhs.shape == (Nlocations + 1,))

#print(constraint_rhs)

(16775,)
Correct shape? True


In [ ]:
# Importing Constraint Matrix

A_ub = sparse.load_npz("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_mat_grad_sparse.npz")

print(A_ub.shape)
print("Correct shape?",A_ub.shape == (Nlocations , Nlocations * 2))

(16774, 33548)
Correct shape? True


In [ ]:
# resetting limit
pc_to_place = 100

total_limit = np.round(current_AED_num * pc_to_place / 100)
#total_limit = 100 # alternative manual setting

print("AEDs to place:", total_limit)

constraint_rhs[0] = total_limit

AEDs to place: 7926.0


In [ ]:
# Performing linear programming

solution = linprog(c = objective_fct,
                   A_ub = A_ub,
                   A_eq = np.matrix(1-objective_fct),
                   b_ub = - constraint_rhs[1:],
                   b_eq = constraint_rhs[0:1],
                   bounds = (0,15),
                   integrality= 1-objective_fct
                   )

print(solution.message)
print("success:", solution.success)
print("status:", solution.status)

In [ ]:
# Checking solution

print(solution.message)
print(max(solution.x[:Nlocations]))
print(max(solution.x[Nlocations+1:]))
print("Checking if placed correct number: ", (sum(solution.x[:Nlocations]) == total_limit)) # should be true

solution_name = "gradated_sol_" + str(round(total_limit)) + "AEDs_" + str(Nlocations) + "locations" + name_ext

np.savetxt("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_exports/" + solution_name + ".csv", solution.x)

# Gradated Demand Rearranging


In [ ]:
# Importing objective function

objective_fct = pd.read_csv('/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/objective_fct.csv')
objective_fct = objective_fct[["x"]].values
objective_fct = objective_fct.squeeze()

print(objective_fct.shape)
print("Correct shape?",objective_fct.shape == (Nlocations * 2,))


#print(objective_fct)

(33548,)
Correct shape? True


In [ ]:
# Importing RHS


constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_rhs_grad_rearrange10pc.csv")

constraint_rhs = constraint_rhs[["x"]].values
constraint_rhs = constraint_rhs.squeeze()

print(constraint_rhs.shape)
print("Correct shape?",constraint_rhs.shape == (Nlocations + 1,))

#print(constraint_rhs)

(16775,)
Correct shape? True


In [ ]:
# Importing Constraint Matrix

A_ub = sparse.load_npz("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_imports/constraint_mat_grad_sparse.npz")

print(A_ub.shape)
print("Correct shape?",A_ub.shape == (Nlocations , Nlocations * 2))

(16774, 33548)
Correct shape? True


In [ ]:
# resetting limit
pc_to_place = 10

total_limit = np.round(current_AED_num * pc_to_place / 100)
#total_limit = 100 # alternative manual setting

print("AEDs to place:", total_limit)

constraint_rhs[0] = total_limit

AEDs to place: 2378.0


In [ ]:
# Performing linear programming

solution = linprog(c = objective_fct,
                   A_ub = A_ub,
                   A_eq = np.matrix(1-objective_fct),
                   b_ub = - constraint_rhs[1:],
                   b_eq = constraint_rhs[0:1],
                   bounds = (0,15),
                   integrality= 1-objective_fct
                   )

print(solution.message)
print("success:", solution.success)
print("status:", solution.status)

In [ ]:
# Checking solution

print(solution.message)
print(max(solution.x[:Nlocations]))
print(max(solution.x[Nlocations+1:]))
print("Checking if placed correct number: ", (sum(solution.x[:Nlocations]) == total_limit)) # should be true

solution_name = "gradated_sol_" + str(round(total_limit)) + "AEDs_" + str(Nlocations) + "locations" + "rearranged10pc"

np.savetxt("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_exports/" + solution_name + ".csv", solution.x)